<a href="https://colab.research.google.com/github/chenharry66/adversial-audio-experiments/blob/main/refactored_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Setup

In [ ]:
#install depencies
!pip install werpy scipy
!pip install --upgrade "numba==0.68.0"
!pip install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 208.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [werpy]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.4 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 94.9 MB/s  0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.44.0
    Uninstalling llvmlite-0.44.0:
      Successfully uninstalled llvmlite-0.44.0
  Attempting uninstall: numba
    Found existing installation: numba 0.61.2
    Uninstalling numba-0.61.2:
      Successfully uninstalled numba-0.61.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.


##Model Definition and Helper Functions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import regex as re
import werpy, torch
import torch.nn.functional as F
from scipy.io import wavfile

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = model.to(device)
model.eval()

def custom_normalize(text):
    clean_text = re.sub(r'[\p{P}\p{S}]', '', text)
    return werpy.normalize(clean_text)

def load_flac(path):
    audio, sample_rate = librosa.load(path, sr=16000, mono=True)
    return audio, sample_rate

def load_wav(path):
    audio, sample_rate = wavfile.read(path)
    return audio, sample_rate

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

Using: cuda


In [ ]:
from pathlib import Path

def index_librispeech(root):
    root = Path(root)
    records = []
    for transcript_path in sorted(root.rglob("*.trans.txt")):
        with transcript_path.open("r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                recording_id, transcript = line.split(maxsplit=1)
                audio_path = transcript_path.parent / f"{recording_id}.flac"

                if not audio_path.is_file():
                    raise FileNotFoundError(f"Missing audio: {audio_path}")

                speaker_id, chapter_id, utterance_id = recording_id.split("-")
                records.append({
                    "id": recording_id,
                    "path": str(audio_path),
                    "reference": transcript,
                    "speaker_id": speaker_id,
                    "chapter_id": chapter_id,
                })

    return records

In [ ]:
import csv
import json
import numpy as np

def save_result(run_dir, recording_number, sample_rate, reference, original_audio, delta_stage1, delta_final, evaluations, history):
    def to_numpy(value):
        if isinstance(value, torch.Tensor):
            value = value.detach().cpu().numpy()
        return np.asarray(value, dtype=np.float32)
    def to_scalar(value):
        if isinstance(value,(torch.Tensor, np.generic)):
            return value.item()
        return value

    original = to_numpy(original_audio)
    delta1 = to_numpy(delta_stage1)
    delta2 = to_numpy(delta_final)

    metadata = {
        "sample_rate": int(sample_rate),
        "num_samples": len(original),
        "duration_seconds": len(original) / sample_rate,
        "reference": custom_normalize(reference),
        "evaluations": {},
    }

    audio_files = {
        "original": "original.wav",
        "stage1": "stage1.wav",
        "modified": "modified.wav",
    }

    for stage, filename in audio_files.items():
        transcription, wer = evaluations[stage]
        metadata["evaluations"][stage] = {
            "audio_file": filename,
            "transcription": transcription,
            "wer": wer,
        }

    folder = Path(run_dir) / f"recording{recording_number:03d}"

    # Refuse to overwrite an existing recording folder.
    folder.mkdir(parents=True, exist_ok=False)

    wavfile.write(folder / "original.wav", int(sample_rate), original)
    wavfile.write(folder / "stage1.wav", int(sample_rate), original + delta1)
    wavfile.write(folder / "modified.wav", int(sample_rate), original + delta2)
    wavfile.write(folder / "perturbation.wav", int(sample_rate), delta2)

    np.save(folder / "delta.npy", delta2, allow_pickle=False)

    with (folder / "result.json").open("w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2, ensure_ascii=False, allow_nan=False)

    fields = ["stage", "step", "delta_rms", "asr_loss", "masking_loss"]

    with (folder / "history.csv").open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for row in history:
            writer.writerow({
                field: to_scalar(row.get(field))
                for field in fields
            })
    return folder

In [ ]:
def compute_log_spec(audio):
    window = torch.hann_window(400, device=audio.device, dtype=audio.dtype)
    stft = torch.stft(audio, n_fft=400, hop_length=160, window=window, return_complex=True)
    magnitudes = (stft[..., :-1].abs() ** 2).contiguous()
    mel_filters = torch.as_tensor(
    processor.feature_extractor.mel_filters, device=audio.device, dtype=audio.dtype)
    mel_spec = mel_filters.T @ magnitudes
    log_spec = mel_spec.clamp_min(1e-10).log10()
    peak = log_spec.amax(dim=(-2, -1), keepdim=True)
    log_spec = torch.maximum(log_spec, peak - 8.0)
    return (log_spec + 4.0) / 4.0

In [ ]:
# figure out how the
def training1(clip, labels, MAX_ITERS=200, ALPHA=1e-1):
    history = []

    audio_rms = clip.square().mean().sqrt()
    target_rms = audio_rms / (10 ** (23 / 20))

    delta = torch.zeros_like(clip, requires_grad=True)
    optimizer = torch.optim.Adam([delta], lr=ALPHA)

    for p in model.parameters():
        p.requires_grad = False

    for i in range(MAX_ITERS):
        optimizer.zero_grad(set_to_none=True
                            )
        perturbation = clip + delta
        whisper_audio = F.pad(perturbation, (0, 480000 - perturbation.shape[-1]))
        log_spec = compute_log_spec(whisper_audio)
        out = model(log_spec.unsqueeze(0), labels=labels)

        loss = -out.loss
        loss.backward()

        with torch.no_grad():
            if i % 10 == 0:
                actual_rms = delta.square().mean().sqrt()
                budget_used = actual_rms / target_rms.clamp_min(1e-12)

                grad_std = delta.grad.std(unbiased=False)
                delta_std = delta.std(unbiased=False)

                print(
                    f"Step {i} | "
                    f"ASR loss={out.loss.item():.4f} | "
                    f"delta RMS={actual_rms.item():.3e} | "
                    f"allowed RMS={target_rms.item():.3e} | "
                    f"budget used={100 * budget_used.item():.2f}%"
                )

                if delta_std.item() > 0:
                    ratio = ALPHA * grad_std / delta_std
                    print(f"Update / delta: {ratio.item():.4f}")

                history.append({
                    "stage": 1,
                    "step": i,
                    "delta_rms": actual_rms.item(),
                    "asr_loss": out.loss.item(),
                    "masking_loss": None,
                })

            optimizer.step()
            current_rms = torch.sqrt(torch.mean(delta**2))
            print(f"Iteration {i} RMS: {current_rms}")
            if current_rms > target_rms:
                delta *= (target_rms / current_rms)
    return delta, history

In [ ]:
# returns a wer
def eval(aug_audio, reference, rate):
    if isinstance(aug_audio, torch.Tensor):
        aug_audio = aug_audio.detach().cpu().numpy()

    inputs = processor(aug_audio, return_tensors="pt", sampling_rate=rate)

    with torch.inference_mode():
        generated_ids = model.generate(input_features=inputs["input_features"].to(device))

    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcription = custom_normalize(transcription)
    reference = custom_normalize(reference)
    print(transcription)
    print(reference)

    wer = werpy.wer(reference, transcription)
    return transcription, float(wer)

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/FOLDER/01")

In [ ]:
import librosa
def compute_PSD_for_perturbation(perturbation, psd_max, window_size):
    window = torch.hann_window(window_size, device=perturbation.device, dtype=perturbation.dtype)
    spectrum = torch.stft(perturbation, window=window, n_fft=window_size, hop_length=window_size // 4, onesided=True, center=False, return_complex=True)
    z = (8 / 3) ** 0.5 * torch.abs(spectrum / window_size)
    psd = 10 * torch.log10(z.square() + 1e-20)
    normalized_psd = 96 - 10 * torch.log10(psd_max + 1e-20) + psd
    return (10 ** (normalized_psd / 10)).T

In [ ]:
def compute_mask_threshold_loss(psd_perturb, theta_xs):
    average_loss_per_frame = torch.mean((psd_perturb - theta_xs).relu(), dim=1)
    return torch.mean(average_loss_per_frame)

In [ ]:
def training2(clip, delta, labels, MAX_ITERS=200, ALPHA=1e-3, BETA=0.5, window_size = 2048):
    history = []
    theta_xs, psd_max = masking.generate_th(clip.detach().cpu().numpy(), 16000, window_size=window_size)
    theta_xs = torch.as_tensor(theta_xs, dtype=clip.dtype, device=clip.device)
    psd_max = torch.as_tensor( psd_max, dtype=clip.dtype, device=clip.device)
    optimizer = torch.optim.Adam([delta], lr=ALPHA)

    for p in model.parameters():
        p.requires_grad = False
    for i in range(MAX_ITERS):
        # delta.grad=None
        optimizer.zero_grad(set_to_none=True)
        perturbation = clip + delta
        whisper_audio = F.pad(perturbation, (0, 480000 - perturbation.shape[-1]))
        log_spec = compute_log_spec(whisper_audio)
        out = model(log_spec.unsqueeze(0), labels=labels)
        psd_perturb = compute_PSD_for_perturbation(delta, psd_max, window_size)
        masking_loss = compute_mask_threshold_loss(psd_perturb, theta_xs)
        loss = -out.loss + BETA * masking_loss
        print(f"Iteration {i} Loss: {loss.item()}")
        if i % 10 == 0:
            history.append({
                "stage": 2,
                "step": i,
                "delta_rms": delta.detach().square().mean().sqrt().item(),
                "asr_loss": out.loss.item(),
                "masking_loss": masking_loss.item()
            })
        loss.backward()
        with torch.no_grad():
            # grad_std = delta.grad.std(unbiased=False)
            # delta_std = delta.std(unbiased=False)
            # audio_std = clip.std(unbiased=False)
            # estimated_update_std = ALPHA * grad_std

            # print(
            #     "Update / delta:",
            #     (estimated_update_std / delta_std.clamp_min(1e-12)).item(),
            # )

            # delta -= ALPHA * delta.grad
            optimizer.step()

    return delta.detach().clone(), history

In [ ]:
import generate_masking_threshold as masking
def optimize_one_recording(path, reference):
    data, rate= load_flac(path)
    clip = torch.as_tensor(data, dtype=torch.float32, device=device)
    labels = processor.tokenizer(reference, return_tensors="pt", return_attention_mask=False)["input_ids"].to(device)

    if not 2048 <= clip.numel() <= 480000:
        raise ValueError("This version supports recordings up to 30 seconds.")

    original_eval = eval(clip, reference, rate)
    delta_net, history1 = training1(clip, labels, MAX_ITERS=2)
    delta_snapshot = delta_net.detach().clone()
    stage1_eval = eval(clip + delta_snapshot, reference, rate)

    delta_mask, history2 = training2(clip, delta_net, labels, MAX_ITERS=2)
    aug_audio = clip + delta_mask
    final_eval= eval(aug_audio, reference, rate)
    evaluations = {
        "original": original_eval,
        "stage1": stage1_eval,
        "modified": final_eval,
    }

    print("BEFORE RETURN")
    for stage, evaluation in evaluations.items():
        print(stage, evaluation)

    return {
    "original_audio": data,
    "sample_rate": rate,
    "reference": reference,
    "delta_stage1": delta_snapshot,
    "delta_final": delta_mask,
    "history": history1 + history2,
    "evaluations": evaluations
    }

In [ ]:
from datetime import datetime
import random

def generate_perturbations(numRecordings):
    output_root = Path("/content/drive/MyDrive/audio_experiments")
    run_id = datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
    run_dir = output_root / run_id
    run_dir.mkdir(parents=True, exist_ok=False)

    root = Path("/content/drive/MyDrive/FOLDER/01/LibriSpeech")
    records = index_librispeech(root)
    selected = random.Random(42).sample(records, k=numRecordings)

    for i, record in enumerate(selected):
        result = optimize_one_recording(record["path"], record["reference"])
        saved={
            "run_dir": run_dir,
            "recording_number": i,
            **result
        }

        save_result(**saved)

In [ ]:
generate_perturbations(1)